<a href="https://colab.research.google.com/github/Polon-Lubvi/ICTIS/blob/main/01_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Система анализа новостного контента: суммаризация и тональность на LoRA

Курсовой проект по ТАБД (вариант 26) + индивидуальное задание «Машинное обучение + Интернет-маркетинг».
Автор: Мамченко В.В., КТмо2-11.

**Назначение.** Дообучение малой открытой языковой модели методом LoRA на корпусе новостей Gazeta: краткое изложение новости и определение её тональности для медиамониторинга брендов.

**Как запустить**
1. Среда выполнения → Сменить среду выполнения → **T4 GPU**.
2. Слева «Секреты» (значок ключа) → добавить `HF_TOKEN` (токен Hugging Face, тип Read) и включить доступ из блокнота.
3. Среда выполнения → **Выполнить все**. При первом запуске разрешить доступ к Google Drive.

**Где данные и результаты:** Google Drive, папка `MyDrive/tabd26` (`data/raw`, `data/processed`, `checkpoints`, `mlruns`, `reports`).
Повторный запуск не скачивает данные и не обучает модели заново, если результат уже есть на Drive.

**Репозиторий:** вставить ссылку на GitHub.

## 0. Установка и настройка

In [ ]:
# A3: после первого успешного запуска замени строку ниже на вывод pip freeze (пакет==версия)
!pip install -q transformers peft trl bitsandbytes accelerate datasets evaluate rouge_score bert_score mlflow pandera gradio sumy datasketch natasha
!pip freeze | grep -iE "^(transformers|peft|trl|bitsandbytes|accelerate|datasets|evaluate|rouge.score|bert.score|mlflow|pandera|gradio|sumy|datasketch|natasha|torch)=="

In [ ]:
import os, time, csv, datetime
import numpy as np, pandas as pd, torch
from transformers import set_seed

SEED = 42
set_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет")

In [ ]:
# A4: токен только из секретов Colab, никогда не печатаем его
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ROOT = "/content/drive/MyDrive/tabd26"
for sub in ["data/raw", "data/processed", "checkpoints", "mlruns", "reports"]:
    os.makedirs(f"{ROOT}/{sub}", exist_ok=True)

In [ ]:
# E1: журнал прогона — время и объём каждого шага
LOG = f"{ROOT}/reports/run_log.csv"

def log_step(step, started, rows_in=None, rows_out=None, skipped=False):
    new = not os.path.exists(LOG)
    with open(LOG, "a", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        if new:
            w.writerow(["step", "started", "seconds", "rows_in", "rows_out", "skipped"])
        w.writerow([step, datetime.datetime.fromtimestamp(started).isoformat(timespec="seconds"),
                    round(time.time() - started, 1), rows_in, rows_out, skipped])

## 1. Данные: загрузка в слой raw (C1)

In [ ]:
from datasets import load_dataset

t0 = time.time()
raw = {}
skipped = True
for split in ["train", "validation", "test"]:
    path = f"{ROOT}/data/raw/gazeta_{split}.parquet"
    if not os.path.exists(path):          # E2: повторный запуск ничего не перезаписывает
        load_dataset("IlyaGusev/gazeta", revision="v2.0", split=split).to_parquet(path)
        skipped = False
    raw[split] = pd.read_parquet(path)
    print(split, raw[split].shape)
log_step("load_raw", t0, rows_out=sum(len(d) for d in raw.values()), skipped=skipped)

## 2. Правила качества (C2) и отчёт (C3)

In [ ]:
try:
    import pandera.pandas as pa      # pandera >= 0.24
except ImportError:
    import pandera as pa

schema = pa.DataFrameSchema(
    {
        "text":    pa.Column(str, [pa.Check.str_length(200, 20000)], nullable=False),
        "summary": pa.Column(str, [pa.Check.str_length(min_value=20)], nullable=False),
        "title":   pa.Column(str, nullable=True),
        "date":    pa.Column(str, nullable=False),
        "url":     pa.Column(str, unique=True, nullable=False),
    },
    checks=[
        pa.Check(lambda df: df["summary"].str.len() < df["text"].str.len(), name="summary_shorter_than_text"),
        pa.Check(lambda df: pd.to_datetime(df["date"], errors="coerce").notna(), name="date_parses"),
    ],
    strict=True,
)

t0 = time.time()
reports = []
for split, df in raw.items():
    try:
        schema.validate(df, lazy=True)
        print(split, "— нарушений нет")
    except pa.errors.SchemaErrors as e:
        fc = e.failure_cases.assign(split=split)
        reports.append(fc)
        print(split, "— нарушений:", len(fc))
quality = pd.concat(reports) if reports else pd.DataFrame()
quality.to_csv(f"{ROOT}/reports/quality.csv", index=False)
log_step("quality", t0, rows_in=sum(len(d) for d in raw.values()), rows_out=len(quality))
quality.groupby(["split", "check"]).size() if len(quality) else "Все правила выполнены"

## 3. Проверка утечки между разбиениями (26.1)

In [ ]:
from datasketch import MinHash, MinHashLSH

def shingles(text, k=5):
    w = text.lower().split()
    return {" ".join(w[i:i+k]) for i in range(max(1, len(w) - k + 1))}

def minhash(text):
    m = MinHash(num_perm=64, seed=SEED)
    for s in shingles(text):
        m.update(s.encode("utf-8"))
    return m

t0 = time.time()
train, test = raw["train"], raw["test"]
exact_url  = len(set(train.url) & set(test.url))
exact_text = len(set(train.text) & set(test.text))

lsh = MinHashLSH(threshold=0.8, num_perm=64)
for i, t in enumerate(train.text):
    lsh.insert(str(i), minhash(t))
near = [j for j, t in enumerate(test.text) if lsh.query(minhash(t))]

dates = {s: pd.to_datetime(d.date, errors="coerce") for s, d in raw.items()}
leak = {
    "exact_url": exact_url,
    "exact_text": exact_text,
    "near_duplicates_test": len(near),
    "train_dates": f"{dates['train'].min().date()} — {dates['train'].max().date()}",
    "test_dates":  f"{dates['test'].min().date()} — {dates['test'].max().date()}",
}
pd.Series(leak).to_csv(f"{ROOT}/reports/leakage.csv")
log_step("leakage", t0, rows_in=len(train) + len(test), rows_out=len(near))
leak

Минхэширование всего train (около 61 тыс. статей) занимает несколько минут на CPU. Для отладки можно взять `train.sample(10000, random_state=SEED)`, но в итоговом прогоне нужен весь train.

## 4. Фиксированная выборка для оценки (26.3)

In [ ]:
EVAL_PATH = f"{ROOT}/data/processed/eval_sample.parquet"
if not os.path.exists(EVAL_PATH):
    raw["test"].sample(500, random_state=SEED).to_parquet(EVAL_PATH)
eval_df = pd.read_parquet(EVAL_PATH)
eval_df.shape

## 5. Базовый метод Lead-3 и метрики (26.1)

In [ ]:
import re, evaluate
from razdel import sentenize

def lead3(text):
    return " ".join(s.text for s in list(sentenize(text))[:3])

rouge = evaluate.load("rouge")
tokenize = lambda s: re.findall(r"\w+", s.lower())   # стандартный токенизатор ROUGE выкидывает кириллицу

preds = [lead3(t) for t in eval_df.text]
scores = rouge.compute(predictions=preds, references=list(eval_df.summary), tokenizer=tokenize)
scores

## Дальше
- TextRank (`sumy`) и zero-shot модель — тот же расчёт метрик.
- MLflow и 4 конфигурации LoRA — шаги 3–4 инструкции.